# House Prices EDA and Preprocessing - 15 Selected Fields

            Notebook này rebuild phase feature selection, EDA và preprocessing cho
            Kaggle House Prices. Scope dừng ở preprocessing: không train model,
            không evaluate model.

In [ ]:
from pathlib import Path

            import matplotlib.pyplot as plt
            import numpy as np
            import pandas as pd
            import seaborn as sns

            PROJECT_DIR = Path.cwd().parents[1] if Path.cwd().name == "house_prices" else Path("../../").resolve()
            DATA_DIR = PROJECT_DIR / "data" / "raw"
            ARTIFACT_DIR = PROJECT_DIR / "artifacts" / "preprocessing"
            TRAIN_PATH = DATA_DIR / "train.csv"
            TEST_PATH = DATA_DIR / "test.csv"
            PREPROCESSOR_PATH = ARTIFACT_DIR / "house_prices_preprocessor.pkl"
            PROCESSED_TRAIN_PATH = ARTIFACT_DIR / "processed_train.csv"
            PROCESSED_TEST_PATH = ARTIFACT_DIR / "processed_test.csv"
            FEATURE_NOTES_PATH = ARTIFACT_DIR / "feature_notes.csv"

            sns.set_theme(style="whitegrid")
            pd.set_option("display.max_columns", 120)

In [ ]:
import sys

            src_path = PROJECT_DIR / "src"
            if str(src_path) not in sys.path:
                sys.path.insert(0, str(src_path))

            from ml.preprocessing import (
                FEATURE_COLUMNS,
                FEATURE_NOTES,
                NOMINAL_FEATURES,
                NUMERIC_FEATURES,
                ORDINAL_FEATURES,
                TARGET_COLUMN,
                fit_transform_house_prices,
                save_feature_notes,
                transform_house_prices,
            )

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
            test_df = pd.read_csv(TEST_PATH)

            print(f"Train shape: {train_df.shape}")
            print(f"Test shape: {test_df.shape}")
            display(train_df[FEATURE_COLUMNS + [TARGET_COLUMN]].head())

## Feature Selection Rationale

            15 fields được chọn để cân bằng tín hiệu numeric mạnh với
            categorical/ordinal có ý nghĩa domain rõ ràng. Numeric features
            được scale; nominal categorical được one-hot; quality fields được
            ordinal encode theo thứ tự chất lượng.

In [ ]:
feature_notes = pd.DataFrame(FEATURE_NOTES)
            display(feature_notes)

## Data Overview and Missing Values

In [ ]:
selected_train = train_df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
            selected_test = test_df[FEATURE_COLUMNS].copy()

            dtype_missing = pd.DataFrame({
                "dtype": selected_train[FEATURE_COLUMNS].dtypes.astype(str),
                "missing_train": selected_train[FEATURE_COLUMNS].isna().sum(),
                "missing_test": selected_test[FEATURE_COLUMNS].isna().sum(),
                "missing_train_pct": selected_train[FEATURE_COLUMNS].isna().mean().round(4),
                "missing_test_pct": selected_test[FEATURE_COLUMNS].isna().mean().round(4),
                "nunique_train": selected_train[FEATURE_COLUMNS].nunique(dropna=True),
            })
            display(dtype_missing)

## Target Distribution

            SalePrice lệch phải rõ rệt; log1p(SalePrice) gần đối xứng hơn nên
            nên dùng cho phase train sau.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.histplot(train_df[TARGET_COLUMN], kde=True, bins=40, ax=axes[0])
            axes[0].set_title("SalePrice distribution")
            sns.histplot(
                np.log1p(train_df[TARGET_COLUMN]),
                kde=True,
                bins=40,
                ax=axes[1],
                color="seagreen",
            )
            axes[1].set_title("log1p(SalePrice) distribution")
            plt.tight_layout()
            print({
                "saleprice_skew": float(train_df[TARGET_COLUMN].skew()),
                "log_saleprice_skew": float(np.log1p(train_df[TARGET_COLUMN]).skew()),
            })

## Numeric Feature EDA

In [ ]:
display(train_df[NUMERIC_FEATURES].describe().T)

            train_df[NUMERIC_FEATURES].hist(figsize=(14, 10), bins=30)
            plt.suptitle("Numeric feature distributions", y=1.02)
            plt.tight_layout()

In [ ]:
corr = train_df[NUMERIC_FEATURES + [TARGET_COLUMN]].corr(numeric_only=True)
            price_corr = corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(
                key=lambda s: s.abs(),
                ascending=False,
            )
            display(price_corr.to_frame("corr_with_saleprice"))

            plt.figure(figsize=(10, 8))
            sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
            plt.title("Correlation matrix: numeric features and SalePrice")
            plt.tight_layout()

## Categorical Feature EDA

In [ ]:
for col in NOMINAL_FEATURES + ORDINAL_FEATURES:
                print(f"\n{col}")
                display(train_df[col].value_counts(dropna=False).to_frame("count"))
                summary = (
                    train_df.groupby(col, dropna=False)[TARGET_COLUMN]
                    .agg(["count", "median", "mean"])
                    .sort_values("median", ascending=False)
                )
                display(summary)

In [ ]:
categorical_cols = NOMINAL_FEATURES + ORDINAL_FEATURES
            fig, axes = plt.subplots(
                len(categorical_cols),
                1,
                figsize=(14, 4 * len(categorical_cols)),
            )
            for ax, col in zip(axes, categorical_cols):
                order = (
                    train_df.groupby(col, dropna=False)[TARGET_COLUMN]
                    .median()
                    .sort_values(ascending=False)
                    .index
                )
                sns.boxplot(data=train_df, x=col, y=TARGET_COLUMN, order=order, ax=ax)
                ax.set_title(f"SalePrice by {col}")
                ax.tick_params(axis="x", rotation=45)
            plt.tight_layout()

## Outlier Detection

            Các điểm diện tích rất lớn nhưng giá thấp cần ghi chú cho phase
            train sau. Notebook này chỉ phát hiện và ghi nhận, chưa drop rows để
            tránh thay đổi dữ liệu trước khi có model experiment.

In [ ]:
outlier_rules = {
                "GrLivArea_gt_4000_low_price": (
                    (train_df["GrLivArea"] > 4000)
                    & (train_df[TARGET_COLUMN] < 300000)
                ),
                "TotalBsmtSF_gt_5000": train_df["TotalBsmtSF"] > 5000,
                "GarageArea_gt_1200": train_df["GarageArea"] > 1200,
                "SalePrice_gt_700000": train_df[TARGET_COLUMN] > 700000,
            }
            outlier_summary = pd.Series(
                {name: int(mask.sum()) for name, mask in outlier_rules.items()},
                name="count",
            )
            display(outlier_summary.to_frame())

            for name, mask in outlier_rules.items():
                if mask.any():
                    display(
                        train_df.loc[
                            mask,
                            ["Id", "GrLivArea", "GarageArea", "TotalBsmtSF", TARGET_COLUMN],
                        ]
                    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
            sns.scatterplot(data=train_df, x="GrLivArea", y=TARGET_COLUMN, ax=axes[0])
            axes[0].set_title("GrLivArea vs SalePrice")
            sns.scatterplot(data=train_df, x="GarageArea", y=TARGET_COLUMN, ax=axes[1])
            axes[1].set_title("GarageArea vs SalePrice")
            sns.scatterplot(data=train_df, x="TotalBsmtSF", y=TARGET_COLUMN, ax=axes[2])
            axes[2].set_title("TotalBsmtSF vs SalePrice")
            plt.tight_layout()

## Preprocessing Export

            Artifacts được xuất để sẵn sàng cho phase train sau: processed
            train/test CSV, fitted preprocessor pkl, và feature notes CSV.
            Không có model training trong notebook này.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
            processed_train = fit_transform_house_prices(
                df=train_df,
                preprocessor_path=PREPROCESSOR_PATH,
                processed_csv_path=PROCESSED_TRAIN_PATH,
            )
            processed_test = transform_house_prices(
                df=test_df,
                preprocessor_path=PREPROCESSOR_PATH,
                processed_csv_path=PROCESSED_TEST_PATH,
            )
            save_feature_notes(FEATURE_NOTES_PATH)

            print(f"Processed train shape: {processed_train.shape}")
            print(f"Processed test shape: {processed_test.shape}")
            print(f"Preprocessor: {PREPROCESSOR_PATH}")
            print(f"Processed train CSV: {PROCESSED_TRAIN_PATH}")
            print(f"Processed test CSV: {PROCESSED_TEST_PATH}")
            print(f"Feature notes CSV: {FEATURE_NOTES_PATH}")
            assert processed_train.drop(columns=[TARGET_COLUMN, "SalePriceLog"]).shape[1] == processed_test.shape[1]
            assert processed_train.isna().sum().sum() == 0
            assert processed_test.isna().sum().sum() == 0
            display(processed_train.head())

## Summary Notes

            - Tín hiệu mạnh nhất: OverallQual, GrLivArea, GarageCars,
              GarageArea, TotalBsmtSF, 1stFlrSF, Neighborhood, ExterQual,
              KitchenQual, BsmtQual.
            - Transform khuyến nghị cho target ở phase train: np.log1p(SalePrice).
              Artifact train đã có thêm SalePriceLog để dùng lại.
            - Numeric features đã scale; nominal categorical đã one-hot; ordinal
              quality features đã ordinal encode.
            - Outliers được ghi nhận nhưng chưa loại bỏ trong phase preprocessing này.